In [ ]:
# ruff: noqa: F401, F403

import os
import subprocess
import sys
import typing as tp

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch

from IPython.display import *

from pacer import (
    CoordinateSystem,
    # read_dat_file,
    DatVersion,
    GPMFSource,
    GPSSample,
    Lap,
    Laps,
    Point,
    PointInTime_GPSSample,
    RawGPSSource,
    ReferenceTrack,
    Segment,
    SequentialGPSSource,
    Vec3f,
)

In [107]:
files = [
    "/Users/denys/Documents/2026-jan-buckmore-park/GX010300.MP4",
    "/Users/denys/Documents/2026-jan-buckmore-park/GX020300.MP4",
    # "/Users/denys/Documents/2025-nov-buckmore-testing/GH010279.MP4",
    # "/Users/denys/Documents/2025-nov-buckmore-testing/GH020279.MP4",
    # "/Users/denys/Documents/2025-nov-buckmore-testing/GH030279.MP4",
    # "/Users/denys/Documents/2025-nov-buckmore-testing/GH040279.MP4",
]

samples = []
file_sources = [GPMFSource(f) for f in files]
sources = [SequentialGPSSource(file_sources[0], file_sources[1])]
for source in file_sources:
    while not source.is_end():
        b, e = source.current_time_span()
        source.read_samples(lambda s, _, _2: samples.append((s, (b, e))))
        source.next()

In [108]:
def fix_samples(
    samples: list[tuple[GPSSample, tuple[float, float]]],
) -> list[GPSSample]: ...

In [109]:
single_files = [GPMFSource(f) for f in files]
intermediate = []

for i in range(len(single_files)):
    if i == 0:
        intermediate.append(single_files[i])
    else:
        intermediate.append(SequentialGPSSource(intermediate[i - 1], single_files[i]))

gpmf = intermediate[-1]

gpmf.get_total_duration()
samples = []


def on_sample(s: GPSSample, _, _2):
    if s.full_speed > 3:
        samples.append((s, gpmf.current_time_span()))


while not gpmf.is_end():
    gpmf.read_samples(on_sample)
    gpmf.next()

No payload


In [110]:
cs = CoordinateSystem(samples[0][0])

In [111]:
px.scatter(
    [
        cs.distance(ps, ss) / (ps.full_speed + ss.full_speed) * 2
        for (ps, _), (ss, _2) in zip(samples[:-1], samples[1:])
    ]
)

In [112]:
def interpolate_timestamps(
    samples: list[GPSSample], cs: CoordinateSystem
) -> np.ndarray:
    rough_timings = np.array(
        [np.nan]
        + [
            cs.distance(ps, ss) / (ps.full_speed + ss.full_speed) * 2
            for ps, ss in zip(samples[:-1], samples[1:])
        ]
    )

    median = np.nanmedian(rough_timings)

    # fig = px.scatter(rough_timings, log_y=True)
    # fig.add_hline(median)
    # fig

    di = (rough_timings / median).round()
    di[np.isnan(di)] = 0

    # px.scatter(di, log_y=True)

    boundaries = np.concat([np.arange(len(di))[di > 3], [len(di)]])
    segments = [
        (int(b), int(e)) for b, e in zip(boundaries + 1, boundaries[1:]) if e - b > 10
    ]
    segments

    ts = np.array([s.timestamp_ms for s in samples])

    for b, e in segments:
        total_shift = di[b + 1 : e].sum()
        t0, tn = samples[b].timestamp_ms, samples[e - 1].timestamp_ms
        ts[b:e] = (
            np.concat([[0], np.cumsum(di[b + 1 : e])]) * (tn - t0) / total_shift + t0
        ).astype(int)

    return ts


ts = interpolate_timestamps([s for s, _ in samples], cs)
px.scatter(ts[1:] - ts[:-1], log_y=True)

In [113]:
laps = Laps()
laps.set_coordinate_system(cs)

start = pd.to_datetime(samples[0][0].timestamp_ms, unit="ms")

for (s, _), t in zip(samples, ts):
    laps.add_point(s, (pd.to_datetime(t, unit="ms") - start).total_seconds())

In [114]:
s = Segment(Point(x=-135, y=-160), Point(x=-125, y=-180))
laps.sectors.start_line = s
laps.update()

laps_times = pd.DataFrame(
    [dict(lap=i, lap_time=laps.lap_time(i)) for i in range(laps.laps_count())]
)
px.line(
    laps_times.where(lambda d: (d["lap_time"] > 10) & (d["lap_time"] < 90)),
    x="lap",
    y="lap_time",
    title="Lap times",
    markers=True,
)

In [115]:
laps_times

,lap,lap_time
0,0,72.013693
1,1,70.503489
2,2,67.260882
3,3,65.192040
4,4,64.953614
5,5,64.274034
6,6,61.539847
7,7,61.120672
8,8,59.674395
9,9,59.743508


In [72]:
start_line = laps.sectors.start_line

In [116]:
fig = px.line(
    x=[cs.local(s).x for s, _ in samples], y=[cs.local(s).y for s, _ in samples]
)
fig.add_trace(
    px.line(
        x=[start_line.first.x, start_line.second.x],
        y=[start_line.first.y, start_line.second.y],
    ).data[0]
)
fig

In [118]:
best_lap = laps_times.loc[
    lambda d: d["lap_time"] > 0.95 * np.median(d["lap_time"]), "lap_time"
].idxmin()
best_lap

np.int64(22)

In [ ]:
delta_by_lap = []
reference_lap = ReferenceTrack.from_lap(laps.get_lap(best_lap), 5, cs)

lap1 = reference_lap.resample(laps.get_lap(best_lap))
t1 = (
    np.array([lap1.points[i].time for i in range(len(lap1.points))])
    - lap1.points[0].time
)

for i in range(0, laps.laps_count() - 1):
    lap0 = reference_lap.resample(laps.get_lap(i))

    t0 = (
        np.array([lap0.points[i].time for i in range(len(lap0.points))])
        - lap0.points[0].time
    )

    try:
        delta = t1 - t0

        delta_by_lap.append(
            pd.DataFrame(
                dict(
                    lat=[lap0.points[i].point.lat for i in range(len(lap0.points))],
                    lon=[lap0.points[i].point.lon for i in range(len(lap0.points))],
                    speed=[
                        lap0.points[i].point.full_speed for i in range(len(lap0.points))
                    ],
                    distance=lap0.cum_distances,
                    t=t0 + lap0.points[0].time,
                    delta=delta,
                    lap=i,
                )
            )
        )
    except ValueError as e:
        print(f"Skipping lap {i} due to error: {e}")

delta_by_lap = pd.concat(delta_by_lap)

In [120]:
fig = px.line(
    delta_by_lap.assign(
        speed=lambda d: d["speed"] * 3.6,
        local_t=lambda d: d["t"] - (d["t"] / 531).astype(int) * 531,
        lap_time=lambda d: pd.merge(d, laps_times, on="lap", how="left")[
            "lap_time"
        ].values,
    ).loc[lambda d: d["lap_time"] < np.median(d["lap_time"]) * 1.07],
    x="distance",
    y="delta",
    hover_data=["local_t", "speed", "lap_time"],
    color="lap",
)
fig.show()

In [121]:
px.line(
    delta_by_lap.assign(speed=lambda d: d["speed"] * 3.6),
    x="distance",
    y="speed",
    color="lap",
)

In [124]:
px.scatter(
    delta_by_lap.assign(
        ddelta=lambda d: d["delta"].diff().rolling(20).mean().clip(-1e-2, +1e-2),
    ).loc[lambda d: d["lap"].isin([26, 28])],
    x="lon",
    y="lat",
    color="ddelta",
    hover_data=["distance", "lap", "delta"],
).update_layout(height=600)

In [88]:
ddelta = delta_by_lap["delta"].diff()
ddelta = (
    ddelta.clip(
        lower=delta.mean() - 2 * ddelta.std(), upper=ddelta.mean() + 2 * ddelta.std()
    )
    .rolling(20)
    .mean()
)


px.scatter_map(
    delta_by_lap.assign(
        ddelta=ddelta, local_t=lambda d: d["t"].where(lambda d: d < 531, d["t"] - 531)
    ).loc[lambda d: d["lap"].isin([2, 23])],
    lat="lat",
    lon="lon",
    color="speed",
    hover_data=["distance", "delta", "lap", "local_t"],
    # map_style="satellite",
    zoom=17,
).update_layout(height=800)

In [44]:
px.histogram(ddelta, nbins=1_000)

In [45]:
px.histogram(delta_by_lap["delta"], nbins=200)

In [91]:
lap = laps.get_lap(1)


def build_lap_df(lap):
    return pd.DataFrame(
        [
            dict(
                lat=s.point.latitude,
                lon=s.point.longitude,
                time=s.time - lap.points[0].time,
                distance=lap.cum_distances[i],
                i_point=i,
            )
            for i in range(lap.count())
            if (s := lap.points[i]) is not None
        ]
    )


all_laps = pd.concat(
    [build_lap_df(laps.get_lap(i)).assign(i_lap=i) for i in range(laps.laps_count())]
)


AttributeError: 'pacer._pacer.GPSSample' object has no attribute 'latitude'

In [ ]:
fig = px.scatter_map(
    all_laps,
    lat="lat",
    lon="lon",
    color="distance",
    hover_data=["i_lap", "i_point"],
    # map_style="basic",
    zoom=17,
)
fig.update_layout(height=800)


In [ ]:
start_line = laps.sectors.start_line
p1, p2 = start_line.first, start_line.second

In [ ]:
s1, s2 = map(lambda p: getattr(cs, "global")(Vec3f(p.x, p.y, 0)), (p1, p2))

In [ ]:
def add_segment(fig: go.Figure, s1: GPSSample, s2: GPSSample, name: str):
    fig.add_trace(
        go.Scattermap(
            mode="lines",
            lon=[s1.longitude, s2.longitude],
            lat=[s1.latitude, s2.latitude],
            name=name,
        )
    )
    return fig


add_segment(fig, s1, s2, "start_line")

In [ ]:
Point(1, 2)

Point(x=1.0, y=2.0)

In [ ]:
start_line

Segment(first=Point(x=83.9594233828202, y=-26.0512637901178), second=Point(x=75.5699800474182, y=-31.49343619845611))

In [ ]:
def to_point(s: GPSSample):
    v = getattr(cs, "local")(s)
    return Point(v[0], v[1])


first_seg = Segment(
    to_point(laps.get_lap(0).points[0].point), to_point(laps.get_lap(0).points[1].point)
)

add_segment(
    fig, laps.get_lap(0).points[0].point, laps.get_lap(0).points[1].point, "first_one"
)

In [ ]:
ratio, _ = (
    start_line.intersects(first_seg.first, first_seg.second),
    start_line.intersects(first_seg.second, first_seg.first),
)

In [ ]:
df = pd.DataFrame(
    dict(
        x=[
            start_line.first.x,
            start_line.second.x,
            None,
            first_seg.first.x,
            first_seg.second.x,
        ],
        y=[
            start_line.first.y,
            start_line.second.y,
            None,
            first_seg.first.y,
            first_seg.second.y,
        ],
        name=[1, 1, None, 2, 3],
    )
)

fig = px.line(df, x="x", y="y", hover_data="name")

In [ ]:
x, y = start_line.first, start_line.second
a, b = first_seg.first, first_seg.second

In [ ]:
res = (1 - ratio) * a + ratio * b
fig.add_trace(go.Scatter(x=[res.x], y=[res.y]))

In [ ]:
def rot(x):
    return Point(-x.y, x.x)

In [ ]:
n = rot(x - y)
n.scalar(a - x), n.scalar(b - x), n.scalar(a - y), n.scalar(b - y)

(-4.940411947284312,
 0.8100397188352676,
 -4.940411947284313,
 0.8100397188352667)

In [ ]:
a

Point(x=79.49651065482897, y=-29.535207990925674)

In [ ]:
dir(start_line.first)

['__add__',
 '__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__mul__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rmul__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__sub__',
 '__subclasshook__',
 'scalar',
 'x',
 'y']

In [ ]:
start_line.first.scalar(start_line.first)

7727.85311983796

In [ ]:
start_line.intersects(first_seg.first, first_seg.second, None)

TypeError: intersects(): incompatible function arguments. The following argument types are supported:
    1. intersects(self, fst: _pacer_geometry_impl.Point, snd: _pacer_geometry_impl.Point) -> float | None

Invoked with types: _pacer_laps.Segment, _pacer_geometry_impl.Point, _pacer_geometry_impl.Point, NoneType

In [ ]:
cs.global_()

AttributeError: '_pacer_geometry_impl.CoordinateSystem' object has no attribute 'global_'

In [ ]:
px.scatter_map(
    points_df,
    lat="latitude",
    lon="longitude",
    color="speed",
    map_style="outdoors",
).update_layout(height=800)

NameError: name 'points_df' is not defined

In [ ]:
lap = laps.get_lap(1)
dir(lap)

In [ ]:
lap.points

In [ ]:
delta_by_lap

In [ ]:
px.scatter(delta[1:] - delta[:-1])

In [ ]:
c = 0.1
noise = np.round((delta[1:] - delta[:-1]) / c) * c
px.scatter(noise)

In [ ]:
px.scatter(np.cumsum(noise), title="Cumulative noise")

In [ ]:
px.line(delta - np.concatenate([[0], np.cumsum(noise)]))

In [ ]:
laps = Laps()
laps.set_coordinate_system(cs)

for (s, span), t in zip(samples, t2):
    laps.add_point(s, t)
s = laps.pick_random_start()
laps.sectors.start_line = s
laps.update()
reference_lap = ReferenceTrack.from_lap(laps.get_lap(1), 5, cs)
lap0 = reference_lap.resample(laps.get_lap(3))
lap1 = reference_lap.resample(laps.get_lap(4))
px.line(
    [
        lap1.points[i].time
        - lap0.points[i].time
        - lap1.points[0].time
        + lap0.points[0].time
        for i in range(len(lap0.points))
    ]
)